# MODULE 4 â€” Nettoyage des Donnees

Dans le monde reel, les donnees sont rarement parfaites.

Il y a souvent des **valeurs manquantes**, des **doublons**, des **erreurs de saisie**, des **dates mal formatees**, des **valeurs aberrantes**...

Le **nettoyage des donnees** (data cleaning) est l'etape la plus importante en data science. On dit souvent que ca prend 80% du temps d'un data scientist !

Dans ce module, tu vas apprendre :
- Identifier les valeurs manquantes avec `isnull().sum()`
- Corriger les valeurs manquantes avec `fillna()`
- Supprimer les doublons avec `drop_duplicates()`
- Nettoyer les textes avec `str.strip().str.title()`
- Convertir les dates avec `pd.to_datetime()`
- Detecter les valeurs aberrantes avec la methode IQR
- Creer une fonction pipeline de nettoyage
- Sauvegarder le dataset propre

In [1]:
# On importe les bibliotheques necessaires
import pandas as pd    # pour manipuler les donnees
import numpy as np     # pour les calculs numeriques

# On charge le dataset
df = pd.read_csv("ventes_koryxa.csv", encoding="utf-8")
print(f"Dataset charge : {df.shape[0]} lignes, {df.shape[1]} colonnes")

Dataset charge : 100 lignes, 13 colonnes


---
## PARTIE 1 â€” Identifier les Valeurs Manquantes

### Qu'est-ce qu'une valeur manquante ?

Une **valeur manquante** (ou valeur nulle) c'est quand une case du tableau est **vide** : on ne connait pas l'information.

Par exemple, si un client n'a pas renseigne son age, cette case sera vide (NaN = "Not a Number" en anglais).

NaN est la facon de Python de dire "Je ne sais pas".

In [2]:
# isnull() cree un tableau True/False : True = valeur manquante
# .sum() compte le nombre de True par colonne
# = nombre de valeurs manquantes par colonne
valeurs_manquantes = df.isnull().sum()

print("=== VALEURS MANQUANTES PAR COLONNE ===")
print(valeurs_manquantes)

=== VALEURS MANQUANTES PAR COLONNE ===
id                0
date              0
client            0
age               5
ville             0
region           22
produit           0
categorie         0
quantite          0
prix_unitaire     0
total             0
paiement          3
satisfaction      6
dtype: int64


In [3]:
# On affiche seulement les colonnes qui ont des valeurs manquantes
# valeurs_manquantes > 0 = seulement les colonnes avec des manquants
colonnes_avec_manquants = valeurs_manquantes[valeurs_manquantes > 0]

print("Colonnes avec des valeurs manquantes :")
print(colonnes_avec_manquants)
print()
# On calcule le pourcentage de valeurs manquantes
for col, nb in colonnes_avec_manquants.items():
    pct = nb / len(df) * 100
    print(f"  {col}: {nb} manquants ({pct:.1f}%)")

Colonnes avec des valeurs manquantes :
age              5
region          22
paiement         3
satisfaction     6
dtype: int64

  age: 5 manquants (5.0%)
  region: 22 manquants (22.0%)
  paiement: 3 manquants (3.0%)
  satisfaction: 6 manquants (6.0%)


---
## PARTIE 2 â€” Corriger les Valeurs Manquantes

### fillna() : remplir les cases vides

`fillna()` remplace les valeurs manquantes (NaN) par une valeur qu'on choisit.

**Quelle valeur mettre a la place ?**
- Pour un **nombre** : souvent la moyenne ou la mediane
- Pour un **texte** : souvent "Inconnu" ou la valeur la plus frequente

In [4]:
# On travaille sur une copie pour ne pas modifier l'original
df_propre = df.copy()   # .copy() cree une copie independante

# Remplir l'age manquant par la mediane de l'age
# La mediane est plus robuste que la moyenne face aux valeurs extremes
age_median = df_propre["age"].median()   # calcule la mediane
print(f"Age median : {age_median}")

df_propre["age"] = df_propre["age"].fillna(age_median)  # remplace NaN par la mediane
print(f"Valeurs manquantes dans 'age' apres correction : {df_propre['age'].isnull().sum()}")

Age median : 43.0
Valeurs manquantes dans 'age' apres correction : 0


In [5]:
# Remplir les valeurs texte manquantes par "Inconnu"
colonnes_texte = ["ville", "region"]  # colonnes texte avec des manquants

for col in colonnes_texte:
    nb_avant = df_propre[col].isnull().sum()   # compte avant
    df_propre[col] = df_propre[col].fillna("Inconnu")  # remplace par "Inconnu"
    nb_apres = df_propre[col].isnull().sum()   # compte apres
    print(f"{col}: {nb_avant} manquants -> {nb_apres} manquants")

ville: 0 manquants -> 0 manquants
region: 22 manquants -> 0 manquants


---
## PARTIE 3 â€” Supprimer les Doublons

### Qu'est-ce qu'un doublon ?

Un **doublon** c'est quand la meme ligne apparait deux fois (ou plus) dans le tableau.

C'est comme avoir la meme vente enregistree deux fois. Ca fausserait tous nos calculs !

`drop_duplicates()` supprime les lignes en double.

In [6]:
# On compte les doublons avant nettoyage
nb_doublons = df_propre.duplicated().sum()   # .duplicated() repere les lignes en double
print(f"Nombre de doublons detectes : {nb_doublons}")

Nombre de doublons detectes : 0


In [7]:
# On supprime les doublons avec drop_duplicates()
# keep="first" = on garde la premiere occurrence, on supprime les suivantes
df_propre = df_propre.drop_duplicates(keep="first")

print(f"Lignes avant : {len(df)}")
print(f"Lignes apres suppression des doublons : {len(df_propre)}")
print(f"Doublons supprimes : {len(df) - len(df_propre)}")

Lignes avant : 100
Lignes apres suppression des doublons : 100
Doublons supprimes : 0


---
## PARTIE 4 â€” Nettoyer les Textes

### Problemes courants dans les textes

Les donnees textuelles ont souvent des problemes :
- **Espaces en trop** : "Paris " au lieu de "Paris"
- **Majuscules inconsistantes** : "paris", "PARIS", "Paris" = 3 valeurs differentes pour la meme ville !
- **Caracteres parasites**

On utilise les methodes `str` de Pandas pour nettoyer.

In [8]:
# Verifier les problemes de casse (majuscules/minuscules)
print("Villes uniques dans les donnees :")
print(sorted(df_propre["ville"].unique()[:10]))  # 10 premieres villes uniques

Villes uniques dans les donnees :
['Abidjan', 'Bruxelles', 'Casablanca', 'Dakar', 'Marseille', 'Montréal', 'Nantes', 'Paris', 'Strasbourg', 'paris']


In [9]:
# str.strip() supprime les espaces au debut et a la fin
# str.title() met la premiere lettre en majuscule, le reste en minuscule
# "paris" -> "Paris", " PARIS " -> "Paris"
df_propre["ville"] = df_propre["ville"].str.strip().str.title()
df_propre["region"] = df_propre["region"].str.strip().str.title()
df_propre["client"] = df_propre["client"].str.strip()
df_propre["produit"] = df_propre["produit"].str.strip()

print("Villes apres nettoyage :")
print(sorted(df_propre["ville"].unique()[:10]))

Villes apres nettoyage :
['Abidjan', 'Bruxelles', 'Casablanca', 'Dakar', 'Lyon', 'Marseille', 'Montréal', 'Nantes', 'Paris', 'Strasbourg']


---
## PARTIE 5 â€” Convertir les Dates

### pd.to_datetime() : transformer les textes en vraies dates

Les dates sont souvent stockees comme du texte dans les CSV.

Par exemple, "2024-10-18" est un texte, pas une vraie date. Python ne peut pas faire de calculs dessus.

`pd.to_datetime()` transforme ces textes en **vrais objets dates** que Python comprend.

In [10]:
# Avant conversion : le type de la colonne date
print("Type avant conversion :", df_propre["date"].dtype)  # object = texte
print("Exemples :", df_propre["date"].head(3).tolist())

Type avant conversion : object
Exemples : ['2024-10-18', '2024-07-18', '2024-11-12']


In [11]:
# pd.to_datetime() convertit les textes en vraies dates
# errors="coerce" = si une date est invalide, on met NaT (date manquante)
df_propre["date"] = pd.to_datetime(df_propre["date"], errors="coerce")

print("Type apres conversion  :", df_propre["date"].dtype)  # datetime64
print("Exemples              :", df_propre["date"].head(3).tolist())

Type apres conversion  : datetime64[ns]
Exemples              : [Timestamp('2024-10-18 00:00:00'), Timestamp('2024-07-18 00:00:00'), Timestamp('2024-11-12 00:00:00')]


In [12]:
# Maintenant on peut extraire des informations de la date
# .dt donne acces aux proprietes de date
df_propre["annee"] = df_propre["date"].dt.year     # l'annee
df_propre["mois"] = df_propre["date"].dt.month     # le mois (1-12)
df_propre["jour"] = df_propre["date"].dt.day       # le jour

print("Nouvelles colonnes extraites de la date :")
print(df_propre[["date", "annee", "mois", "jour"]].head(5))

Nouvelles colonnes extraites de la date :
        date   annee  mois  jour
0 2024-10-18  2024.0  10.0  18.0
1 2024-07-18  2024.0   7.0  18.0
2 2024-11-12  2024.0  11.0  12.0
3 2024-02-05  2024.0   2.0   5.0
4 2024-02-18  2024.0   2.0  18.0


---
## PARTIE 6 â€” Detecter les Valeurs Aberrantes

### Qu'est-ce qu'une valeur aberrante (outlier) ?

Une **valeur aberrante** est une valeur qui s'eloigne beaucoup des autres.

Par exemple, si toutes les ventes sont entre 50 et 500 euros, mais qu'une vente est a 50 000 euros, c'est probablement une erreur.

**Methode IQR (Interquartile Range)** :
- On calcule le "quartile 1" (Q1) = valeur en dessous de laquelle se trouvent 25% des donnees
- On calcule le "quartile 3" (Q3) = valeur en dessous de laquelle se trouvent 75% des donnees
- IQR = Q3 - Q1 (l'ecart entre les deux)
- Tout ce qui est en dessous de Q1 - 1.5*IQR ou au dessus de Q3 + 1.5*IQR est suspect

In [13]:
# Methode IQR pour detecter les outliers sur la colonne "total"
Q1 = df_propre["total"].quantile(0.25)   # 25% des valeurs sont en dessous
Q3 = df_propre["total"].quantile(0.75)   # 75% des valeurs sont en dessous
IQR = Q3 - Q1                             # ecart interquartile

# Limites de detection
borne_basse = Q1 - 1.5 * IQR   # en dessous = trop bas
borne_haute = Q3 + 1.5 * IQR   # au dessus = trop haut

print(f"Q1 (25%) = {Q1:.2f} euros")
print(f"Q3 (75%) = {Q3:.2f} euros")
print(f"IQR = {IQR:.2f} euros")
print(f"Borne basse : {borne_basse:.2f} euros")
print(f"Borne haute : {borne_haute:.2f} euros")

Q1 (25%) = 894.54 euros
Q3 (75%) = 2916.70 euros
IQR = 2022.16 euros
Borne basse : -2138.71 euros
Borne haute : 5949.95 euros


In [14]:
# On identifie les outliers
outliers = df_propre[(df_propre["total"] < borne_basse) | (df_propre["total"] > borne_haute)]
print(f"Nombre d'outliers detectes : {len(outliers)}")

if len(outliers) > 0:
    print("Exemples d'outliers :")
    print(outliers[["client", "produit", "total"]].head(5))

Nombre d'outliers detectes : 5
Exemples d'outliers :
            client            produit     total
6      Laura Petit           Écran 4K  27827.19
13  Mohamed Traoré  Clavier Mécanique  73427.97
22    Alice Martin           Écran 4K  26774.45
46   Georges Mbeki          Webcam HD  93668.05
60     Hana Yamada  Clavier Mécanique   9301.34


---
## PARTIE 7 â€” Fonction Pipeline de Nettoyage

### Creer une fonction qui fait tout en une fois

On va maintenant tout regrouper dans une **fonction pipeline**.

Un **pipeline** c'est comme une chaine d'usine : les donnees entrent sales, passent par chaque etape de nettoyage, et sortent propres.

On peut ensuite appeler cette fonction sur n'importe quel dataset similaire !

In [15]:
# Fonction complete de nettoyage
def nettoyer_dataset(df_brut):
    """
    Nettoie le dataset de ventes Koryxa.
    Etapes : copie, valeurs manquantes, doublons, textes, dates
    Retourne le dataset nettoye.
    """
    # ETAPE 1 : Copie pour ne pas modifier l'original
    df_clean = df_brut.copy()

    # ETAPE 2 : Valeurs manquantes
    df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())
    df_clean["ville"] = df_clean["ville"].fillna("Inconnu")
    df_clean["region"] = df_clean["region"].fillna("Inconnue")

    # ETAPE 3 : Doublons
    df_clean = df_clean.drop_duplicates(keep="first")

    # ETAPE 4 : Nettoyage textes
    df_clean["ville"] = df_clean["ville"].str.strip().str.title()
    df_clean["region"] = df_clean["region"].str.strip().str.title()
    df_clean["client"] = df_clean["client"].str.strip()
    df_clean["produit"] = df_clean["produit"].str.strip()

    # ETAPE 5 : Conversion des dates
    df_clean["date"] = pd.to_datetime(df_clean["date"], errors="coerce")
    df_clean["annee"] = df_clean["date"].dt.year
    df_clean["mois"] = df_clean["date"].dt.month

    return df_clean

print("Fonction nettoyer_dataset() definie et prete !")

Fonction nettoyer_dataset() definie et prete !


In [16]:
# On applique la fonction sur le dataset original
df_original = pd.read_csv("ventes_koryxa.csv", encoding="utf-8")
df_nettoye = nettoyer_dataset(df_original)

print("=== RAPPORT DE NETTOYAGE ===")
print(f"Lignes avant : {len(df_original)}")
print(f"Lignes apres : {len(df_nettoye)}")
print(f"Valeurs manquantes restantes : {df_nettoye.isnull().sum().sum()}")
print(f"Colonnes : {list(df_nettoye.columns)}")

=== RAPPORT DE NETTOYAGE ===
Lignes avant : 100
Lignes apres : 100
Valeurs manquantes restantes : 27
Colonnes : ['id', 'date', 'client', 'age', 'ville', 'region', 'produit', 'categorie', 'quantite', 'prix_unitaire', 'total', 'paiement', 'satisfaction', 'annee', 'mois']


In [17]:
# ETAPE FINALE : Sauvegarde du dataset propre
# C'est ce fichier qui sera utilise par les modules 5 et 6
df_nettoye.to_csv("ventes_koryxa_propre.csv", index=False, encoding="utf-8")

print("Fichier 'ventes_koryxa_propre.csv' sauvegarde avec succes !")
print(f"Dimensions : {df_nettoye.shape[0]} lignes x {df_nettoye.shape[1]} colonnes")

Fichier 'ventes_koryxa_propre.csv' sauvegarde avec succes !
Dimensions : 100 lignes x 15 colonnes


---
## RECAPITULATIF DU MODULE 4

| Etape | Commande | Ce que ca fait |
|-------|---------|----------------|
| 1. Detecter manquants | `df.isnull().sum()` | Compte les cases vides par colonne |
| 2. Remplir manquants | `df["col"].fillna(valeur)` | Remplace NaN par une valeur |
| 3. Supprimer doublons | `df.drop_duplicates()` | Enleve les lignes en double |
| 4. Nettoyer textes | `str.strip().str.title()` | Supprime espaces, normalise casse |
| 5. Convertir dates | `pd.to_datetime()` | Transforme texte en vraie date |
| 6. Detecter outliers | Methode IQR | Detecte les valeurs extremes |
| 7. Sauvegarder | `df.to_csv("fichier.csv")` | Ecrit le dataset propre |

**Dans le prochain module, on apprend a visualiser les donnees avec des graphiques !**

---
---
# 📖 Ressources — Module 4 : Nettoyage de Données

Pour aller plus loin sur le nettoyage de données, voici 3 ressources en français :

---

### 🔗 1. Nettoyage de données avec Pandas — MonCoachData.com
**Lien :** https://moncoachdata.com/blog/nettoyage-donnees-pandas/

Tutoriel dédié au data cleaning : valeurs manquantes, doublons, types de données.
Très pratique, avec des exemples proches de ce qu'on fait dans ce module.

---

### 🔗 2. Data Cleaning : 7 techniques — Wild Code School
**Lien :** https://www.wildcodeschool.com/fr-FR/blog/data-cleaning-7-techniques-python

Article très pratique qui présente 7 techniques concrètes de nettoyage.
Idéal pour avoir une checklist complète avant de commencer une analyse.

---

### 🔗 3. Nettoyer un jeu de données avec Pandas — CNAM
**Lien :** https://python.sdv.univ-paris-diderot.fr/15_pandas/

Ressource académique française sérieuse, produite par une université.
Avec notebook interactif — tu peux pratiquer directement en ligne.

---

> 💡 **Rappel important :** En data science, on estime que **80% du temps** d'un analyste est consacré au nettoyage des données. Maîtriser ce module, c'est maîtriser 80% du métier.

---
---
# 🎬 Vidéos YouTube — Pour aller plus loin

Ces 2 vidéos en français complètent parfaitement ce module. Regarde-les **après** avoir terminé le notebook pour consolider ce que tu as appris.

---

### 🎥 Vidéo 1 — Data Cleaning avec Python et Pandas : Le Guide Complet

**Rechercher sur YouTube :** [Data Cleaning avec Python et Pandas : Le Guide Complet](https://www.youtube.com/results?search_query=Data%20Cleaning%20avec%20Python%20et%20Pandas%20%3A%20Le%20Guide%20Complet)

1 heure de cours entièrement dédié au nettoyage de données de A à Z. Ce tutoriel va plus loin que notre module et couvre des cas avancés que tu rencontreras en entreprise.

---

### 🎥 Vidéo 2 — Nettoyer et analyser votre jeu de données avec Python — Épisode 1 : Introduction

**Rechercher sur YouTube :** [Nettoyer et analyser votre jeu de données avec Python — Épisode 1 : Introduction](https://www.youtube.com/results?search_query=Nettoyer%20et%20analyser%20votre%20jeu%20de%20donn%C3%A9es%20avec%20Python%20%E2%80%94%20%C3%89pisode%201%20%3A%20Introduction)

Série pratique sur les valeurs manquantes, erreurs et types. Le format en épisodes te permet de regarder à ton rythme et de cibler les points qui te posent problème.

---

> 💡 **Conseil :** Regarde d'abord le notebook en entier, fais les exercices, **puis** regarde les vidéos. Les vidéos servent à consolider, pas à remplacer la pratique.